# Data Audit

Проверка данных перед моделированием оттока: структура, типы, целевая переменная.

80 000 клиентов, 26 признаков.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

df = pd.read_csv("../data/raw/bank_churn_dataset.csv")
print("Размер данных:", df.shape)
print("\nТипы данных:")
print(df.dtypes)

In [ ]:
df.head(10)

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
df.to_sql("bank_churn", engine, if_exists="replace", index=False)
print(f"Загружено {len(df)} строк в PostgreSQL")

In [ ]:
query = """
SELECT
    customer_segment,
    COUNT(*) AS total,
    ROUND(AVG(CAST(exit AS INT)) * 100, 2) AS churn_rate_pct
FROM bank_churn
GROUP BY customer_segment
ORDER BY churn_rate_pct DESC
"""
print("Churn rate по сегментам (SQL-запрос):")
print(pd.read_sql(query, engine))

## Описательная статистика

Минимумы и максимумы — быстрый способ поймать аномалии без отдельных проверок по каждому полю.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().round(2).T

## Распределение целевой переменной

Целевая — `exit`. Если классы сильно несбалансированы, при моделировании понадобится `class_weight='balanced'`.

In [ ]:
churn_rate = df["exit"].mean()
print(f"Общий churn rate: {churn_rate:.2%}")
print(f"\nРаспределение по сегментам:")
print(df.groupby("customer_segment")["exit"].agg(["count", "mean"]).rename(columns={"mean": "churn_rate"}).round(4))

## Итоги аудита

- **Размер:** 80 000 клиентов, 26 признаков.
- **Типы:** числовые + строковые поля, даты в `str` — конвертируются по мере необходимости.
- **Пропуски и дубликаты:** отсутствуют.
- **Диапазоны:** проверены через `describe()` — аномалий нет.
- **Churn rate:** 18% по всей выборке, но крайне неравномерно по сегментам:

| Сегмент | Клиентов | Churn rate |
|---|---|---|
| Mass | 21 436 | **39.5%** |
| Emerging | 31 662 | 17.6% |
| Affluent | 11 210 | 3.1% |
| Priority | 15 692 | **0.07%** |

Mass — 27% клиентской базы, но даёт больше половины всех уходов. Разница с Priority — в 560 раз. Ключевая гипотеза для следующих ноутбуков: что именно в Mass-сегменте приводит к оттоку.

Данные готовы к анализу.